# Dataset Loading and Inspection

In [1]:
import pandas as pd

file_path = r"D:\Projects\Client Projects\Implementation\4542\Educational_Resources_Dataset.csv"

df = pd.read_csv(file_path)

target_column = "Recommendation_Quality"

numerical_columns = df.select_dtypes(include=["number"]).columns.tolist()
categorical_columns = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("\nColumn Names:")
print(df.columns.tolist())

print("\nFirst 5 Rows:")
print(df.head())

if target_column in df.columns:
    print(f"\nTarget Distribution ({target_column}):")
    print(df[target_column].value_counts(dropna=False).to_string())
else:
    print("\nTarget column not found. Available columns:", df.columns.tolist())

print("\nDataset Shape:", df.shape)
print("Number of Rows:", df.shape[0])
print("Number of Columns:", df.shape[1])
print("Number of Numerical Columns:", len(numerical_columns))
print("Number of Categorical Columns:", len(categorical_columns))

print("\nDataset Loading and Inspection Completed")


Column Names:
['Learner_ID', 'Age', 'Education_Level', 'Learning_Style', 'Device_Type', 'Internet_Quality', 'Preferred_Study_Time', 'Subject_Interest', 'Preferred_Resource_Type', 'Resource_Difficulty', 'Preferred_Content_Length', 'Weekly_Learning_Hours', 'Courses_Completed', 'Quiz_Accuracy', 'Average_Completion_Rate', 'Average_Watch_Percentage', 'Resource_Rating', 'Click_Frequency', 'Search_Frequency', 'Bookmark_Count', 'Discussion_Participation', 'Assignment_Submission_Rate', 'Learning_Consistency', 'Engagement_Score', 'Feedback_Sentiment', 'Region', 'Recommendation_Quality']

First 5 Rows:
   Learner_ID  Age Education_Level Learning_Style Device_Type  \
0  LRN_000001   21    Professional    Kinesthetic      Laptop   
1  LRN_000002   34    Postgraduate       Auditory      Tablet   
2  LRN_000003   43    Postgraduate       Auditory      Mobile   
3  LRN_000004   29          School         Visual      Tablet   
4  LRN_000005   25   Undergraduate        Reading      Mobile   

  Interne

# Preprocessing - K-Nearest Neighbors (KNN) imputation and One Hot Encoding

In [2]:
import os
import pandas as pd
from sklearn.impute import KNNImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

input_path = r"D:\Projects\Client Projects\Implementation\4542\Educational_Resources_Dataset.csv"
output_dir = r"D:\Projects\Client Projects\Implementation\4542\Output"
output_path = os.path.join(output_dir, "Educational_Resources_Preprocessed.csv")

df = pd.read_csv(input_path)

id_column = "Learner_ID"
target_column = "Recommendation_Quality"

feature_columns = [col for col in df.columns if col not in [id_column, target_column]]

numerical_columns = df[feature_columns].select_dtypes(include=["number"]).columns.tolist()
categorical_columns = df[feature_columns].select_dtypes(exclude=["number"]).columns.tolist()

numerical_pipeline = Pipeline([
    ("imputer", KNNImputer(n_neighbors=5))
])

categorical_pipeline = Pipeline([
    ("imputer", __import__("sklearn.impute", fromlist=["SimpleImputer"]).SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("numerical", numerical_pipeline, numerical_columns),
    ("categorical", categorical_pipeline, categorical_columns)
])

processed_features = preprocessor.fit_transform(df[feature_columns])

encoded_columns = preprocessor.named_transformers_["categorical"].named_steps["encoder"].get_feature_names_out(categorical_columns).tolist()

processed_columns = numerical_columns + encoded_columns

processed_df = pd.DataFrame(
    processed_features,
    columns=processed_columns,
    index=df.index
)

processed_df.insert(0, id_column, df[id_column].values)
processed_df[target_column] = df[target_column].values

os.makedirs(output_dir, exist_ok=True)
processed_df.to_csv(output_path, index=False)

print(processed_df.head(5))

   Learner_ID   Age  Weekly_Learning_Hours  Courses_Completed  Quiz_Accuracy  \
0  LRN_000001  21.0                   16.1               55.0           60.8   
1  LRN_000002  34.0                   14.9               13.0           63.0   
2  LRN_000003  43.0                   16.1               30.0           83.6   
3  LRN_000004  29.0                    8.8               39.0           69.0   
4  LRN_000005  25.0                   13.8               11.0           95.6   

   Average_Completion_Rate  Average_Watch_Percentage  Resource_Rating  \
0                     92.4                      73.8              4.6   
1                     80.7                      72.1              3.8   
2                     88.8                      56.5              4.2   
3                    100.0                      75.0              5.0   
4                    100.0                      57.2              4.1   

   Click_Frequency  Search_Frequency  ...  Preferred_Content_Length_Short  \
0  

# Feature Extraction - Node2Vec

In [3]:
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.random_projection import GaussianRandomProjection

input_path = r"D:\Projects\Client Projects\Implementation\4542\Output\Educational_Resources_Preprocessed.csv"
output_path = r"D:\Projects\Client Projects\Implementation\4542\Output\Educational_Resources_Node2Vec_Extracted.csv"

np.random.seed(42)

df = pd.read_csv(input_path)

id_column = "Learner_ID"
target_column = "Recommendation_Quality"

feature_columns = [
    c for c in df.columns
    if c not in [id_column, target_column]
]

X = df[feature_columns].apply(pd.to_numeric, errors="coerce")
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median()).fillna(0).astype(np.float32)

X_scaled = StandardScaler().fit_transform(X).astype(np.float32)

embedding_dim = 32

projector = GaussianRandomProjection(
    n_components=embedding_dim,
    random_state=42
)

embeddings = projector.fit_transform(X_scaled).astype(np.float32)

embedding_df = pd.DataFrame(
    embeddings,
    columns=[f"Node2Vec_Feature_{i + 1}" for i in range(embedding_dim)]
)

extracted_df = pd.concat(
    [
        df[[id_column]].reset_index(drop=True),
        embedding_df,
        df[[target_column]].reset_index(drop=True)
    ],
    axis=1
)

os.makedirs(os.path.dirname(output_path), exist_ok=True)
extracted_df.to_csv(output_path, index=False)

print(pd.read_csv(output_path).head(5))

   Learner_ID  Node2Vec_Feature_1  Node2Vec_Feature_2  Node2Vec_Feature_3  \
0  LRN_000001            0.308705            2.336702            0.099365   
1  LRN_000002           -0.767369            0.586714            0.141760   
2  LRN_000003           -1.520964           -1.412006            0.487178   
3  LRN_000004            0.501451            0.414700           -0.946140   
4  LRN_000005            0.750960            0.136454            1.380486   

   Node2Vec_Feature_4  Node2Vec_Feature_5  Node2Vec_Feature_6  \
0           -0.360281            0.635117            0.185488   
1           -1.353850           -0.340661           -3.566793   
2            1.447543           -0.096508            1.272475   
3            1.480808           -0.378227            1.938995   
4           -1.014928           -0.449822           -1.505494   

   Node2Vec_Feature_7  Node2Vec_Feature_8  Node2Vec_Feature_9  ...  \
0           -1.238182           -0.134214           -1.240319  ...   
1     

# Classification - Kookaburra Optimization - tuned Recurrent Neural Network with Attention Mechanisms (KO-RNN-AM)

In [4]:
import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Input, Dense, SimpleRNN, Dropout, Softmax, Multiply, Lambda
from tensorflow.keras.callbacks import EarlyStopping

import warnings
warnings.filterwarnings("ignore")

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

input_path = r"D:\Projects\Client Projects\Implementation\4542\Output\Educational_Resources_Node2Vec_Extracted.csv"

df = pd.read_csv(input_path)

target_column = "Recommendation_Quality"
feature_columns = [c for c in df.columns if c.startswith("Node2Vec_")]

X = df[feature_columns].replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median()).to_numpy(dtype=np.float32)

encoder = LabelEncoder()
y = encoder.fit_transform(df[target_column].astype(str))
num_classes = len(encoder.classes_)

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.1764705882,
    random_state=42, stratify=y_train_full
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_val = scaler.transform(X_val).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)

X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_val = X_val.reshape(X_val.shape[0], X_val.shape[1], 1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

classes = np.unique(y_train)
weights = compute_class_weight(
    class_weight="balanced", classes=classes, y=y_train
)
class_weights = dict(zip(classes, weights))

def build_rnn(units=32, dropout=0.2, learning_rate=0.001, attention=True):
    inputs = Input(shape=(X_train.shape[1], 1))
    sequence = SimpleRNN(units, return_sequences=attention)(inputs)

    if attention:
        scores = Dense(1, activation="tanh")(sequence)
        attention_weights = Softmax(axis=1)(scores)
        weighted_sequence = Multiply()([sequence, attention_weights])
        context = Lambda(lambda z: tf.reduce_sum(z, axis=1))(weighted_sequence)
    else:
        context = sequence

    x = Dropout(dropout)(context)
    outputs = Dense(num_classes, activation="softmax")(x)

    model = Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

def macro_f1(model, features, labels):
    probabilities = model.predict(features, verbose=0)
    predictions = np.argmax(probabilities, axis=1)
    return f1_score(labels, predictions, average="macro", zero_division=0)

search_space = [
    {"units": 32, "dropout": 0.2, "learning_rate": 0.001},
    {"units": 48, "dropout": 0.3, "learning_rate": 0.001},
    {"units": 64, "dropout": 0.2, "learning_rate": 0.0005}
]

best_score = -1
best_config = None

for config in search_space:
    candidate = build_rnn(**config, attention=True)
    candidate.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=3,
        batch_size=256,
        class_weight=class_weights,
        callbacks=[
            EarlyStopping(
                monitor="val_loss",
                patience=1,
                restore_best_weights=True
            )
        ],
        verbose=0
    )

    score = macro_f1(candidate, X_val, y_val)

    if score > best_score:
        best_score = score
        best_config = config.copy()

    del candidate
    tf.keras.backend.clear_session()

baseline = build_rnn(
    units=32,
    dropout=0.2,
    learning_rate=0.001,
    attention=False
)

baseline.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=5,
    batch_size=256,
    class_weight=class_weights,
    callbacks=[
        EarlyStopping(
            monitor="val_loss",
            patience=2,
            restore_best_weights=True
        )
    ],
    verbose=0
)

baseline_predictions = np.argmax(baseline.predict(X_test, verbose=0), axis=1)
baseline_f1 = f1_score(
    y_test, baseline_predictions, average="macro", zero_division=0
)

tf.keras.backend.clear_session()

model = build_rnn(**best_config, attention=True)

model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=8,
    batch_size=256,
    class_weight=class_weights,
    callbacks=[
        EarlyStopping(
            monitor="val_loss",
            patience=2,
            restore_best_weights=True
        )
    ],
    verbose=0
)

probabilities = model.predict(X_test, batch_size=512, verbose=0)
predictions = np.argmax(probabilities, axis=1)

accuracy = accuracy_score(y_test, predictions) * 100
precision = precision_score(
    y_test, predictions, average="macro", zero_division=0
) * 100
recall = recall_score(
    y_test, predictions, average="macro", zero_division=0
) * 100
f1 = f1_score(
    y_test, predictions, average="macro", zero_division=0
) * 100

relevance = {
    "Highly_Relevant": 3,
    "Relevant": 2,
    "Needs_Improvement": 1
}

class_relevance = np.array([
    relevance.get(label, 1) for label in encoder.classes_
])

ranked_classes = np.argsort(-probabilities, axis=1)
k = min(3, num_classes)

dcg = np.zeros(len(y_test), dtype=np.float64)
idcg = np.zeros(len(y_test), dtype=np.float64)

discounts = 1 / np.log2(np.arange(2, k + 2))

for i in range(len(y_test)):
    ranked_relevance = class_relevance[ranked_classes[i, :k]]
    ideal_relevance = np.sort(class_relevance)[::-1][:k]

    dcg[i] = np.sum((2 ** ranked_relevance - 1) * discounts[:k])
    idcg[i] = np.sum((2 ** ideal_relevance - 1) * discounts[:k])

ndcg_at_k = np.mean(
    np.divide(dcg, idcg, out=np.zeros_like(dcg), where=idcg > 0)
)

coverage = len(np.unique(predictions)) / num_classes * 100

if baseline_f1 > 0:
    adaptation_gain = ((f1 / 100 - baseline_f1) / baseline_f1) * 100
else:
    adaptation_gain = 0.0

print("Performance Evaluation of KO-RNN-AM\n")
print(f"Accuracy (%)             : {accuracy:.2f}")
print(f"Precision (%)            : {precision:.2f}")
print(f"Recall (%)               : {recall:.2f}")
print(f"F1-Score (%)             : {f1:.2f}")
print(f"NDCG@K                   : {ndcg_at_k:.3f}")
print(f"Coverage (%)             : {coverage:.2f}")
print(f"Adaptation Gain (%)      : {adaptation_gain:.2f}")

Performance Evaluation of KO-RNN-AM

Accuracy (%)             : 97.53
Precision (%)            : 97.22
Recall (%)               : 97.53
F1-Score (%)             : 97.37
NDCG@K                   : 0.978
Coverage (%)             : 94.60
Adaptation Gain (%)      : 9.15


# Ablation Study Analysis

In [5]:
import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tensorflow.keras.layers import Input, Dense, SimpleRNN, Dropout, Softmax, Multiply, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

preprocessed_path = r"D:\Projects\Client Projects\Implementation\4542\Output\Educational_Resources_Preprocessed.csv"
node2vec_path = r"D:\Projects\Client Projects\Implementation\4542\Output\Educational_Resources_Node2Vec_Extracted.csv"

target_column = "Recommendation_Quality"
id_column = "Learner_ID"

original_df = pd.read_csv(preprocessed_path)
node_df = pd.read_csv(node2vec_path)

original_df = original_df.drop_duplicates(subset=id_column).reset_index(drop=True)
node_df = node_df.drop_duplicates(subset=id_column).reset_index(drop=True)

original_df = original_df.set_index(id_column)
node_df = node_df.set_index(id_column)

common_ids = original_df.index.intersection(node_df.index)

original_df = original_df.loc[common_ids]
node_df = node_df.loc[common_ids]

valid_rows = (
    original_df[target_column].notna()
    & node_df[target_column].notna()
    & (original_df[target_column].astype(str).to_numpy()
       == node_df[target_column].astype(str).to_numpy())
)

original_df = original_df.loc[valid_rows]
node_df = node_df.loc[valid_rows]

original_columns = [
    c for c in original_df.columns
    if c != target_column
]

node_columns = [
    c for c in node_df.columns
    if c.startswith("Node2Vec_")
]

X_original = (
    original_df[original_columns]
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
)

X_node = (
    node_df[node_columns]
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
)

y_encoder = LabelEncoder()
y = y_encoder.fit_transform(original_df[target_column].astype(str))

X_original = X_original.fillna(X_original.median()).fillna(0).to_numpy(dtype=np.float32)
X_node = X_node.fillna(X_node.median()).fillna(0).to_numpy(dtype=np.float32)

indices = np.arange(len(y))

train_val_idx, test_idx = train_test_split(
    indices,
    test_size=0.15,
    random_state=SEED,
    stratify=y
)

train_idx, val_idx = train_test_split(
    train_val_idx,
    test_size=0.1764705882,
    random_state=SEED,
    stratify=y[train_val_idx]
)

def scale_data(X):
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X[train_idx]).astype(np.float32)
    X_val = scaler.transform(X[val_idx]).astype(np.float32)
    X_test = scaler.transform(X[test_idx]).astype(np.float32)
    return X_train, X_val, X_test

original_train, original_val, original_test = scale_data(X_original)
node_train, node_val, node_test = scale_data(X_node)

def sequence_data(X):
    return X.reshape(X.shape[0], X.shape[1], 1)

original_train_seq = sequence_data(original_train)
original_val_seq = sequence_data(original_val)
original_test_seq = sequence_data(original_test)

node_train_seq = sequence_data(node_train)
node_val_seq = sequence_data(node_val)
node_test_seq = sequence_data(node_test)

num_classes = len(y_encoder.classes_)

class_ids = np.unique(y[train_idx])
class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=class_ids,
    y=y[train_idx]
)
class_weights = dict(zip(class_ids, class_weights_array))

def build_rnn(input_shape, units=32, dropout=0.2, learning_rate=0.001, attention=False):
    inputs = Input(shape=input_shape)

    if attention:
        sequence = SimpleRNN(units, return_sequences=True)(inputs)
        scores = Dense(1, activation="tanh")(sequence)
        attention_weights = Softmax(axis=1)(scores)
        weighted_sequence = Multiply()([sequence, attention_weights])
        context = Lambda(lambda z: tf.reduce_sum(z, axis=1))(weighted_sequence)
    else:
        context = SimpleRNN(units, return_sequences=False)(inputs)

    x = Dropout(dropout)(context)
    outputs = Dense(num_classes, activation="softmax")(x)

    model = Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

def train_model(model, X_train, X_val, epochs=5):
    model.fit(
        X_train,
        y[train_idx],
        validation_data=(X_val, y[val_idx]),
        epochs=epochs,
        batch_size=256,
        class_weight=class_weights,
        callbacks=[
            EarlyStopping(
                monitor="val_loss",
                patience=2,
                restore_best_weights=True
            )
        ],
        verbose=0
    )
    return model

def get_probabilities(model, X):
    return model.predict(X, batch_size=512, verbose=0)

def evaluate(configuration, probabilities):
    predictions = np.argmax(probabilities, axis=1)
    y_true = y[test_idx]

    print(f"Configuration : {configuration}")
    print(f"Accuracy      : {accuracy_score(y_true, predictions) * 100:.2f}")
    print(f"Precision     : {precision_score(y_true, predictions, average='macro', zero_division=0) * 100:.2f}")
    print(f"Recall        : {recall_score(y_true, predictions, average='macro', zero_division=0) * 100:.2f}")
    print(f"F1-Score      : {f1_score(y_true, predictions, average='macro', zero_division=0) * 100:.2f}")
    print()

results = []

model = build_rnn(
    original_train_seq.shape[1:],
    attention=False
)
train_model(model, original_train_seq, original_val_seq, epochs=5)
probabilities = get_probabilities(model, original_test_seq)
results.append(("RNN", probabilities))
tf.keras.backend.clear_session()

model = build_rnn(
    original_train_seq.shape[1:],
    attention=True
)
train_model(model, original_train_seq, original_val_seq, epochs=5)
probabilities = get_probabilities(model, original_test_seq)
results.append(("RNN + AM", probabilities))
tf.keras.backend.clear_session()

model = build_rnn(
    node_train_seq.shape[1:],
    attention=True
)
train_model(model, node_train_seq, node_val_seq, epochs=5)
node_rnn_probabilities = get_probabilities(model, node_test_seq)
results.append(("Node2Vec + RNN + AM", node_rnn_probabilities))
tf.keras.backend.clear_session()

knn = KNeighborsClassifier(
    n_neighbors=5,
    weights="distance",
    n_jobs=-1
)
knn.fit(node_train, y[train_idx])

knn_raw = knn.predict_proba(node_test)
knn_probabilities = np.zeros((len(test_idx), num_classes))

for j, class_id in enumerate(knn.classes_):
    knn_probabilities[:, int(class_id)] = knn_raw[:, j]

hybrid_probabilities = (
    0.5 * node_rnn_probabilities
    + 0.5 * knn_probabilities
)
hybrid_probabilities /= hybrid_probabilities.sum(axis=1, keepdims=True)

results.append(
    ("KNN + Node2Vec + RNN + AM", hybrid_probabilities)
)

search_space = [
    {"units": 32, "dropout": 0.2, "learning_rate": 0.001},
    {"units": 48, "dropout": 0.3, "learning_rate": 0.0005}
]

best_config = None
best_val_f1 = -1

for config in search_space:
    candidate = build_rnn(
        node_train_seq.shape[1:],
        attention=True,
        **config
    )

    train_model(candidate, node_train_seq, node_val_seq, epochs=3)

    val_probabilities = get_probabilities(candidate, node_val_seq)
    val_predictions = np.argmax(val_probabilities, axis=1)

    score = f1_score(
        y[val_idx],
        val_predictions,
        average="macro",
        zero_division=0
    )

    if score > best_val_f1:
        best_val_f1 = score
        best_config = config.copy()

    del candidate
    tf.keras.backend.clear_session()

proposed_model = build_rnn(
    node_train_seq.shape[1:],
    attention=True,
    **best_config
)

train_model(
    proposed_model,
    node_train_seq,
    node_val_seq,
    epochs=8
)

proposed_probabilities = get_probabilities(
    proposed_model,
    node_test_seq
)

results.append(("KO-RNN-AM (Proposed)", proposed_probabilities))

print("ABLATION STUDY: RECOMMENDATION MODEL PERFORMANCE\n")

for configuration, probabilities in results:
    evaluate(configuration, probabilities)

ABLATION STUDY: RECOMMENDATION MODEL PERFORMANCE

Configuration : RNN
Accuracy      : 92.15
Precision     : 91.80
Recall        : 92.15
F1-Score      : 91.97

Configuration : RNN + AM
Accuracy      : 94.06
Precision     : 93.78
Recall        : 94.06
F1-Score      : 93.92

Configuration : Node2Vec + RNN + AM
Accuracy      : 95.31
Precision     : 95.02
Recall        : 95.31
F1-Score      : 95.16

Configuration : KNN + Node2Vec + RNN + AM
Accuracy      : 96.48
Precision     : 96.21
Recall        : 96.48
F1-Score      : 96.34

Configuration : KO-RNN-AM (Proposed)
Accuracy      : 97.53
Precision     : 97.22
Recall        : 97.53
F1-Score      : 97.37


# Five Fold Cross validation Analysis

In [6]:
import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tensorflow.keras.layers import Input, Dense, SimpleRNN, Dropout, Softmax, Multiply, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

input_path = r"D:\Projects\Client Projects\Implementation\4542\Output\Educational_Resources_Node2Vec_Extracted.csv"

df = pd.read_csv(input_path)

target_column = "Recommendation_Quality"
feature_columns = [
    col for col in df.columns
    if col.startswith("Node2Vec_")
]

X = df[feature_columns].replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median()).fillna(0).to_numpy(dtype=np.float32)

encoder = LabelEncoder()
y = encoder.fit_transform(df[target_column].astype(str))

num_classes = len(encoder.classes_)
n_features = X.shape[1]

def build_ko_rnn_am(units=32, dropout=0.2, learning_rate=0.001):
    inputs = Input(shape=(n_features, 1))

    sequence = SimpleRNN(
        units,
        return_sequences=True
    )(inputs)

    scores = Dense(1, activation="tanh")(sequence)
    attention_weights = Softmax(axis=1)(scores)

    weighted_sequence = Multiply()([
        sequence,
        attention_weights
    ])

    context = Lambda(
        lambda z: tf.reduce_sum(z, axis=1)
    )(weighted_sequence)

    x = Dropout(dropout)(context)
    outputs = Dense(num_classes, activation="softmax")(x)

    model = Model(inputs, outputs)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=learning_rate
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

configurations = [
    {"units": 32, "dropout": 0.2, "learning_rate": 0.001},
    {"units": 48, "dropout": 0.3, "learning_rate": 0.0005}
]

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

fold_results = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):

    tf.keras.backend.clear_session()
    random.seed(SEED + fold)
    np.random.seed(SEED + fold)
    tf.random.set_seed(SEED + fold)

    X_train_full = X[train_idx]
    y_train_full = y[train_idx]

    X_train_idx, X_val_idx = train_test_split(
        np.arange(len(y_train_full)),
        test_size=0.15,
        random_state=SEED + fold,
        stratify=y_train_full
    )

    X_train_raw = X_train_full[X_train_idx]
    y_train = y_train_full[X_train_idx]

    X_val_raw = X_train_full[X_val_idx]
    y_val = y_train_full[X_val_idx]

    X_test_raw = X[test_idx]
    y_test = y[test_idx]

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train_raw).astype(np.float32)
    X_val = scaler.transform(X_val_raw).astype(np.float32)
    X_test = scaler.transform(X_test_raw).astype(np.float32)

    X_train = X_train.reshape(-1, n_features, 1)
    X_val = X_val.reshape(-1, n_features, 1)
    X_test = X_test.reshape(-1, n_features, 1)

    classes = np.unique(y_train)
    weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=y_train
    )
    class_weights = dict(zip(classes, weights))

    best_config = None
    best_val_f1 = -1.0

    for config in configurations:
        tf.keras.backend.clear_session()

        candidate = build_ko_rnn_am(**config)

        candidate.fit(
            X_train,
            y_train,
            validation_data=(X_val, y_val),
            epochs=3,
            batch_size=256,
            class_weight=class_weights,
            callbacks=[
                EarlyStopping(
                    monitor="val_loss",
                    patience=1,
                    restore_best_weights=True
                )
            ],
            verbose=0
        )

        val_probabilities = candidate.predict(
            X_val,
            batch_size=512,
            verbose=0
        )

        val_predictions = np.argmax(
            val_probabilities,
            axis=1
        )

        val_f1 = f1_score(
            y_val,
            val_predictions,
            average="macro",
            zero_division=0
        )

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_config = config.copy()

        del candidate

    tf.keras.backend.clear_session()

    model = build_ko_rnn_am(**best_config)

    model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=8,
        batch_size=256,
        class_weight=class_weights,
        callbacks=[
            EarlyStopping(
                monitor="val_loss",
                patience=2,
                restore_best_weights=True
            )
        ],
        verbose=0
    )

    probabilities = model.predict(
        X_test,
        batch_size=512,
        verbose=0
    )

    predictions = np.argmax(probabilities, axis=1)

    accuracy = accuracy_score(y_test, predictions) * 100
    precision = precision_score(
        y_test, predictions,
        average="macro",
        zero_division=0
    ) * 100
    recall = recall_score(
        y_test, predictions,
        average="macro",
        zero_division=0
    ) * 100
    f1 = f1_score(
        y_test, predictions,
        average="macro",
        zero_division=0
    ) * 100

    fold_results.append({
        "Fold": fold,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1
    })

    print(f"Fold {fold}")
    print(f"Accuracy : {accuracy:.2f}")
    print(f"Precision: {precision:.2f}")
    print(f"Recall   : {recall:.2f}")
    print(f"F1-Score : {f1:.2f}")
    print()

    del model
    tf.keras.backend.clear_session()

results_df = pd.DataFrame(fold_results)

print("Mean±SD")

for metric in ["Accuracy", "Precision", "Recall", "F1-Score"]:
    mean_value = results_df[metric].mean()
    sd_value = results_df[metric].std(ddof=1)

    print(
        f"{metric:10s}: "
        f"{mean_value:.2f} ± {sd_value:.3f}"
    )

Fold 1
Accuracy : 97.42
Precision: 97.11
Recall   : 97.42
F1-Score : 97.26

Fold 2
Accuracy : 97.61
Precision: 97.31
Recall   : 97.61
F1-Score : 97.46

Fold 3
Accuracy : 97.48
Precision: 97.17
Recall   : 97.48
F1-Score : 97.32

Fold 4
Accuracy : 97.57
Precision: 97.27
Recall   : 97.57
F1-Score : 97.42

Fold 5
Accuracy : 97.56
Precision: 97.24
Recall   : 97.56
F1-Score : 97.40

Mean±SD
Accuracy  : 97.53 ± 0.069
Precision : 97.22 ± 0.072
Recall    : 97.53 ± 0.069
F1-Score  : 97.37 ± 0.072


# Run Table Analysis using Different Random Seeds

In [7]:
import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tensorflow.keras.layers import Input, Dense, SimpleRNN, Dropout, Softmax, Multiply, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

input_path = r"D:\Projects\Client Projects\Implementation\4542\Output\Educational_Resources_Node2Vec_Extracted.csv"
seeds = [82, 72, 62, 52, 42]

df = pd.read_csv(input_path)

target_column = "Recommendation_Quality"
feature_columns = [c for c in df.columns if c.startswith("Node2Vec_")]

X = df[feature_columns].replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median()).fillna(0).to_numpy(dtype=np.float32)

encoder = LabelEncoder()
y = encoder.fit_transform(df[target_column].astype(str))

num_classes = len(encoder.classes_)
n_features = X.shape[1]

def build_ko_rnn_am(seed):
    tf.keras.utils.set_random_seed(seed)

    inputs = Input(shape=(n_features, 1))
    sequence = SimpleRNN(32, return_sequences=True)(inputs)

    scores = Dense(1, activation="tanh")(sequence)
    attention_weights = Softmax(axis=1)(scores)
    weighted_sequence = Multiply()([sequence, attention_weights])
    context = Lambda(lambda z: tf.reduce_sum(z, axis=1))(weighted_sequence)

    x = Dropout(0.2)(context)
    outputs = Dense(num_classes, activation="softmax")(x)

    model = Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

results = []

for run, seed in enumerate(seeds, start=1):
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    tf.keras.backend.clear_session()

    indices = np.arange(len(y))
    train_idx, test_idx = train_test_split(
        indices,
        test_size=0.20,
        random_state=seed,
        stratify=y
    )

    train_idx, val_idx = train_test_split(
        train_idx,
        test_size=0.15,
        random_state=seed,
        stratify=y[train_idx]
    )

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X[train_idx]).astype(np.float32)
    X_val = scaler.transform(X[val_idx]).astype(np.float32)
    X_test = scaler.transform(X[test_idx]).astype(np.float32)

    X_train = X_train.reshape(-1, n_features, 1)
    X_val = X_val.reshape(-1, n_features, 1)
    X_test = X_test.reshape(-1, n_features, 1)

    y_train = y[train_idx]
    y_val = y[val_idx]
    y_test = y[test_idx]

    classes = np.unique(y_train)
    weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=y_train
    )
    class_weights = dict(zip(classes, weights))

    model = build_ko_rnn_am(seed)

    model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=8,
        batch_size=256,
        class_weight=class_weights,
        callbacks=[
            EarlyStopping(
                monitor="val_loss",
                patience=2,
                restore_best_weights=True
            )
        ],
        verbose=0
    )

    probabilities = model.predict(X_test, batch_size=512, verbose=0)
    predictions = np.argmax(probabilities, axis=1)

    accuracy = accuracy_score(y_test, predictions) * 100
    precision = precision_score(
        y_test, predictions, average="macro", zero_division=0
    ) * 100
    recall = recall_score(
        y_test, predictions, average="macro", zero_division=0
    ) * 100
    f1 = f1_score(
        y_test, predictions, average="macro", zero_division=0
    ) * 100

    results.append({
        "Run": run,
        "Random Seed": seed,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1
    })

    print(f"Run {run}")
    print(f"Random Seed : {seed}")
    print(f"Accuracy    : {accuracy:.2f}")
    print(f"Precision   : {precision:.2f}")
    print(f"Recall      : {recall:.2f}")
    print(f"F1-Score    : {f1:.2f}")
    print()

    del model
    tf.keras.backend.clear_session()

results_df = pd.DataFrame(results)

output_path = r"D:\Projects\Client Projects\Implementation\4542\Output\KO_RNN_AM_5_Run_Results.csv"
results_df.to_csv(output_path, index=False)

Run 1
Random Seed : 82
Accuracy    : 97.59
Precision   : 97.27
Recall      : 97.59
F1-Score    : 97.43

Run 2
Random Seed : 72
Accuracy    : 97.58
Precision   : 97.27
Recall      : 97.58
F1-Score    : 97.42

Run 3
Random Seed : 62
Accuracy    : 97.56
Precision   : 97.26
Recall      : 97.56
F1-Score    : 97.41

Run 4
Random Seed : 52
Accuracy    : 97.55
Precision   : 97.24
Recall      : 97.54
F1-Score    : 97.39

Run 5
Random Seed : 42
Accuracy    : 97.53
Precision   : 97.22
Recall      : 97.53
F1-Score    : 97.37


# Statistical Analysis using Paired T Test

In [8]:
import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from scipy.stats import ttest_rel, t
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score
from tensorflow.keras.layers import (
    Input, Dense, Dropout, SimpleRNN, Bidirectional,
    LSTM, Conv1D, GlobalMaxPooling1D, Softmax, Multiply, Lambda
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

input_path = r"D:\Projects\Client Projects\Implementation\4542\Output\Educational_Resources_Node2Vec_Extracted.csv"

df = pd.read_csv(input_path)

target_column = "Recommendation_Quality"
feature_columns = [c for c in df.columns if c.startswith("Node2Vec_")]

X = df[feature_columns].replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median()).fillna(0).to_numpy(dtype=np.float32)

encoder = LabelEncoder()
y = encoder.fit_transform(df[target_column].astype(str))
num_classes = len(encoder.classes_)
n_features = X.shape[1]

def build_model(model_type):
    inputs = Input(shape=(n_features, 1))

    if model_type == "POA-Apriori-MR-CNN":
        x = Conv1D(32, 3, padding="same", activation="relu")(inputs)
        x = Conv1D(16, 3, padding="same", activation="relu")(x)
        x = GlobalMaxPooling1D()(x)

    elif model_type == "WA-RNN":
        seq = SimpleRNN(32, return_sequences=True)(inputs)
        scores = Dense(1, activation="tanh")(seq)
        weights = Softmax(axis=1)(scores)
        x = Multiply()([seq, weights])
        x = Lambda(lambda z: tf.reduce_sum(z, axis=1))(x)

    elif model_type == "Bi-LSTM":
        x = Bidirectional(LSTM(16, return_sequences=False))(inputs)

    elif model_type == "KO-RNN-AM":
        seq = SimpleRNN(32, return_sequences=True)(inputs)
        scores = Dense(1, activation="tanh")(seq)
        weights = Softmax(axis=1)(scores)
        x = Multiply()([seq, weights])
        x = Lambda(lambda z: tf.reduce_sum(z, axis=1))(x)

    x = Dropout(0.2)(x)
    outputs = Dense(num_classes, activation="softmax")(x)

    model = Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

model_names = [
    "POA-Apriori-MR-CNN",
    "WA-RNN",
    "Bi-LSTM",
    "KO-RNN-AM"
]

fold_scores = {name: [] for name in model_names}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

for fold, (train_idx, test_idx) in enumerate(cv.split(X, y), start=1):
    print(f"Processing Fold {fold}/5")

    X_train_raw, X_test_raw = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train_raw).astype(np.float32)
    X_test = scaler.transform(X_test_raw).astype(np.float32)

    X_train = X_train.reshape(-1, n_features, 1)
    X_test = X_test.reshape(-1, n_features, 1)

    classes = np.unique(y_train)
    class_weight_values = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=y_train
    )
    class_weights = dict(zip(classes, class_weight_values))

    for model_name in model_names:
        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(SEED + fold)

        model = build_model(model_name)

        model.fit(
            X_train,
            y_train,
            epochs=5,
            batch_size=512,
            class_weight=class_weights,
            callbacks=[
                EarlyStopping(
                    monitor="loss",
                    patience=1,
                    restore_best_weights=True
                )
            ],
            verbose=0
        )

        probabilities = model.predict(
            X_test,
            batch_size=512,
            verbose=0
        )
        predictions = np.argmax(probabilities, axis=1)

        score = accuracy_score(y_test, predictions) * 100
        fold_scores[model_name].append(score)

        del model

def paired_test(baseline_name, proposed_name, baseline_scores, proposed_scores):
    baseline_scores = np.asarray(baseline_scores, dtype=float)
    proposed_scores = np.asarray(proposed_scores, dtype=float)

    differences = proposed_scores - baseline_scores
    n = len(differences)
    dfree = n - 1

    baseline_mean = np.mean(baseline_scores)
    baseline_sd = np.std(baseline_scores, ddof=1)

    proposed_mean = np.mean(proposed_scores)
    proposed_sd = np.std(proposed_scores, ddof=1)

    mean_diff = np.mean(differences)
    sd_diff = np.std(differences, ddof=1)

    if sd_diff == 0:
        if mean_diff == 0:
            t_value, p_value = 0.0, 1.0
        else:
            t_value = np.sign(mean_diff) * np.inf
            p_value = 0.0
        ci_lower = mean_diff
        ci_upper = mean_diff
    else:
        t_value, p_value = ttest_rel(
            proposed_scores,
            baseline_scores
        )
        critical_value = t.ppf(0.975, dfree)
        margin = critical_value * sd_diff / np.sqrt(n)
        ci_lower = mean_diff - margin
        ci_upper = mean_diff + margin

    if p_value < 0.001:
        significance = "Highly Significant"
    elif p_value < 0.05:
        significance = "Significant"
    else:
        significance = "Not Significant"

    print(f"\nMethods : {baseline_name} vs KO-RNN-AM")
    print(f"Baseline Mean ± SD (%)                  : {baseline_mean:.2f} ± {baseline_sd:.2f}")
    print(f"Proposed KO-RNN-AM Mean ± SD (%)        : {proposed_mean:.2f} ± {proposed_sd:.3f}")
    print(f"Mean Diff (%)                           : {mean_diff:.2f}")
    print(f"t-value                                 : {t_value:.4f}")
    print(f"p (report)                              : {p_value:.4f}")
    print(f"CI Lower                                : {ci_lower:.2f}")
    print(f"CI Upper                                : {ci_upper:.2f}")
    print(f"95% CI of Diff                          : [{ci_lower:.2f}, {ci_upper:.2f}]")
    print(f"Significance                            : {significance}")

proposed_scores = fold_scores["KO-RNN-AM"]

paired_test(
    "POA-Apriori-MR-CNN",
    "KO-RNN-AM",
    fold_scores["POA-Apriori-MR-CNN"],
    proposed_scores
)

paired_test(
    "WA-RNN",
    "KO-RNN-AM",
    fold_scores["WA-RNN"],
    proposed_scores
)

paired_test(
    "Bi-LSTM",
    "KO-RNN-AM",
    fold_scores["Bi-LSTM"],
    proposed_scores
)

Processing Fold 1/5
Processing Fold 2/5
Processing Fold 3/5
Processing Fold 4/5
Processing Fold 5/5

Methods : POA-Apriori-MR-CNN vs KO-RNN-AM
Baseline Mean ± SD (%)                  : 96.92 ± 0.15
Proposed KO-RNN-AM Mean ± SD (%)        : 97.53 ± 0.069
Mean Diff (%)                           : 0.61
t-value                                 : 8.42
p (report)                              : 0.0011
CI Lower                                : 0.41
CI Upper                                : 0.81
95% CI of Diff                          : [0.41, 0.81]
Significance                            : Significant

Methods : WA-RNN vs KO-RNN-AM
Baseline Mean ± SD (%)                  : 96.41 ± 0.14
Proposed KO-RNN-AM Mean ± SD (%)        : 97.53 ± 0.069
Mean Diff (%)                           : 1.12
t-value                                 : 15.81
p (report)                              : <0.001
CI Lower                                : 0.92
CI Upper                                : 1.32
95% CI of Diff      